In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


output_notebook()
hv.extension('bokeh')
pn.extension('bokeh')

font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

Loading BokehJS ...

In [13]:
monkey = 'yasmin' # 'yasmin'  or 'fiona' 
base_path = Path.cwd().parent / 'data' / 'csst_trials_pkls'
filepath = base_path / f'all_{monkey}_CSST_trials_df.pkl'
df = pd.read_pickle(filepath)

print(df.info())
# df.iloc[:2]
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123178 entries, 0 to 123177
Data columns (total 28 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   blinks                  27876 non-null   object 
 1   dir                     123178 non-null  int64  
 2   direction               123178 non-null  object 
 3   filename                123178 non-null  object 
 4   first_relevant_saccade  118561 non-null  object 
 5   flags                   123178 non-null  int64  
 6   go_cue                  123178 non-null  int64  
 7   hPos                    123178 non-null  object 
 8   hVel                    123178 non-null  object 
 9   neural_data             122191 non-null  object 
 10  reaction_time           118561 non-null  float64
 11  saccades                122889 non-null  object 
 12  screen_rotation         123178 non-null  float64
 13  segs_durations          123178 non-null  object 
 14  segs_times          

,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel
0,None,0,R,ya230528a.0525,"[1523, 1598]",8206,1420,"[-12.35, -12.35, -12.35, -12.35, -12.35, -12.3...","[2.7566941723485194, 2.7566941723485194, 3.032...","{1: [90.65, 1290.95, 1364.25, 1872.33, 1935.37...",...,NaN,NaN,False,2571,GO_R,0525,ya230528a,GO,"[-0.45, -0.45, -0.45, -0.475, -0.475, -0.5, -0...","[2.0215757263889143, 2.0215757263889143, 2.113..."
1,None,180,L,ya230528a.0476,"[1680, 1753]",8206,1375,"[-0.15, -0.15, -0.175, -0.175, -0.175, -0.175,...","[-15.161817947916859, -15.161817947916859, -2....","{0: [1394.35, 2074.33], 1: [183.48, 232.45, 27...",...,NaN,NaN,False,2526,GO_L,0476,ya230528a,GO,"[1.1, 1.1, 1.075, 1.075, 1.075, 1.05, 1.075, 1...","[-28.3020601694448, -28.3020601694448, -13.048..."
2,None,0,R,ya230528a.1504,NaN,11278,1276,"[-0.225, -0.225, -0.225, -0.225, -0.225, -0.32...","[0.7351184459596053, 0.7351184459596053, 0.459...","{2: [69.68, 358.55, 504.93, 597.0, 661.85, 832...",...,1.0,1324.0,False,2024,STOP_R_SSD1,1504,ya230528a,STOP,"[-0.175, -0.175, -0.175, -0.175, -0.175, -0.17...","[-2.8485839780934703, -2.8485839780934703, -2...."
3,"[509, 568]",0,R,ya230528a.1499,"[1531, 1606]",8194,1344,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.7890577619318915, 5.7890577619318915, 3.308...","{2: [3.38, 436.4, 554.13, 910.75, 937.85, 1007...",...,NaN,NaN,True,2406,GO_R,1499,ya230528a,GO,"[-0.4, -0.4, -0.4, -0.4, -0.4, -0.45, -0.45, -...","[-0.5513388344697039, -0.5513388344697039, -0...."
4,None,180,L,ya230528a.1105,"[1775, 1851]",8206,1662,"[2.525, 2.525, 2.525, 2.4, 2.4, 2.3, 2.3, 2.3,...","[-39.32883685883888, -39.32883685883888, -39.3...","{1: [678.0, 845.65, 1164.3, 1219.3, 1269.05, 1...",...,NaN,NaN,False,2813,GO_L,1105,ya230528a,GO,"[-7.3, -7.3, -7.3, -6.25, -6.25, -5.625, -5.25...","[188.098432359914, 188.098432359914, 188.09843..."


In [14]:
# Drop unnecessary columns
cols_to_drop = [
    'vPos', 'hPos', 'vVel', 'hVel', 'speed',
    'set', 'direction'
]
df.drop(columns=cols_to_drop, inplace=True)
print(f"DataFrame shape after dropping columns: {df.shape}")
df.head()

DataFrame shape after dropping columns: (123178, 21)


,blinks,dir,filename,first_relevant_saccade,flags,go_cue,neural_data,reaction_time,saccades,screen_rotation,...,segs_times,ssd_len,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type
0,None,0,ya230528a.0525,"[1523, 1598]",8206,1420,"{1: [90.65, 1290.95, 1364.25, 1872.33, 1935.37...",103.0,"[[84, 173], [293, 358], [590, 644], [616, 651]...",0.0,...,"[0, 700, 1420, 1870, 2570]",450,NaN,NaN,False,2571,GO_R,0525,ya230528a,GO
1,None,180,ya230528a.0476,"[1680, 1753]",8206,1375,"{0: [1394.35, 2074.33], 1: [183.48, 232.45, 27...",305.0,"[[0, 16], [1680, 1753], [2017, 2064]]",0.0,...,"[0, 700, 1375, 1825, 2525]",450,NaN,NaN,False,2526,GO_L,0476,ya230528a,GO
2,None,0,ya230528a.1504,NaN,11278,1276,"{2: [69.68, 358.55, 504.93, 597.0, 661.85, 832...",NaN,NaN,0.0,...,"[0, 700, 1276, 1324, 2024]",48,1.0,1324.0,False,2024,STOP_R_SSD1,1504,ya230528a,STOP
3,"[509, 568]",0,ya230528a.1499,"[1531, 1606]",8194,1344,"{2: [3.38, 436.4, 554.13, 910.75, 937.85, 1007...",187.0,"[[209, 264], [468, 534], [502, 537], [551, 606...",0.0,...,"[0, 700, 1344, 1794, 2494]",450,NaN,NaN,True,2406,GO_R,1499,ya230528a,GO
4,None,180,ya230528a.1105,"[1775, 1851]",8206,1662,"{1: [678.0, 845.65, 1164.3, 1219.3, 1269.05, 1...",113.0,"[[0, 72], [1450, 1510], [1775, 1851]]",0.0,...,"[0, 700, 1662, 2112, 2812]",450,NaN,NaN,False,2813,GO_L,1105,ya230528a,GO


In [15]:
# reorder columns
new_order = [
    'filename', 'trial_name', 'reaction_time', 
    'go_cue', 'stop_cue', 'trial_failed', 
    'first_relevant_saccade', 'segs_durations', 'segs_times',
    'trial_length', 'ssd_len', 'ssd_number',
    'screen_rotation', 'neural_data', 'saccades', 
    'blinks', 'dir', 'flags',
    'type', 'trial_session', 'trial_number',
]

df = df[new_order]
df.head()

,filename,trial_name,reaction_time,go_cue,stop_cue,trial_failed,first_relevant_saccade,segs_durations,segs_times,trial_length,...,ssd_number,screen_rotation,neural_data,saccades,blinks,dir,flags,type,trial_session,trial_number
0,ya230528a.0525,GO_R,103.0,1420,NaN,False,"[1523, 1598]","[700, 720, 450, 700]","[0, 700, 1420, 1870, 2570]",2571,...,NaN,0.0,"{1: [90.65, 1290.95, 1364.25, 1872.33, 1935.37...","[[84, 173], [293, 358], [590, 644], [616, 651]...",None,0,8206,GO,ya230528a,0525
1,ya230528a.0476,GO_L,305.0,1375,NaN,False,"[1680, 1753]","[700, 675, 450, 700]","[0, 700, 1375, 1825, 2525]",2526,...,NaN,0.0,"{0: [1394.35, 2074.33], 1: [183.48, 232.45, 27...","[[0, 16], [1680, 1753], [2017, 2064]]",None,180,8206,GO,ya230528a,0476
2,ya230528a.1504,STOP_R_SSD1,NaN,1276,1324.0,False,NaN,"[700, 576, 48, 700]","[0, 700, 1276, 1324, 2024]",2024,...,1.0,0.0,"{2: [69.68, 358.55, 504.93, 597.0, 661.85, 832...",NaN,None,0,11278,STOP,ya230528a,1504
3,ya230528a.1499,GO_R,187.0,1344,NaN,True,"[1531, 1606]","[700, 644, 450, 700]","[0, 700, 1344, 1794, 2494]",2406,...,NaN,0.0,"{2: [3.38, 436.4, 554.13, 910.75, 937.85, 1007...","[[209, 264], [468, 534], [502, 537], [551, 606...","[509, 568]",0,8194,GO,ya230528a,1499
4,ya230528a.1105,GO_L,113.0,1662,NaN,False,"[1775, 1851]","[700, 962, 450, 700]","[0, 700, 1662, 2112, 2812]",2813,...,NaN,0.0,"{1: [678.0, 845.65, 1164.3, 1219.3, 1269.05, 1...","[[0, 72], [1450, 1510], [1775, 1851]]",None,180,8206,GO,ya230528a,1105


In [16]:
# load monkey's cell db from xlsx file
cell_db_path = Path.cwd().parent / 'data' / f'database_sst'
cell_db = pd.read_excel(cell_db_path / f'SST_{monkey}_cells_db.xlsx')
print(cell_db.info())
cell_db.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3773 entries, 0 to 3772
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cell_ID              3773 non-null   int64  
 1   session              3773 non-null   object 
 2   cell_type            3773 non-null   object 
 3   electrode            3773 non-null   int64  
 4   template             3773 non-null   int64  
 5   maestro_ID           3773 non-null   int64  
 6   phy_id               0 non-null      float64
 7   phy_channel          0 non-null      float64
 8   file_begin           3773 non-null   int64  
 9   file_end             3773 non-null   int64  
 10  fb_after_stablility  3773 non-null   object 
 11  fe_after_stability   3773 non-null   object 
 12  plexon_session       3773 non-null   object 
 13  grade                3773 non-null   int64  
 14  X                    3773 non-null   int64  
 15  Y                    3773 non-null   i

,cell_ID,session,cell_type,electrode,template,maestro_ID,phy_id,phy_channel,file_begin,file_end,...,X,Y,depth_mm,is_continuous,comments,plex_sorted_file,tmp,sorted,problem,synced_stability
0,2001,ya230501,msn,1,1,1,NaN,NaN,915,1418,...,-3,-2,9400,2,depth micro m in trec system inlcuding 1000 in...,ya230501a-02.pl2,NaN,1,NaN,1
1,2002,ya230501,msn,1,2,2,NaN,NaN,915,1418,...,-3,-2,9400,2,depth micro m in trec system inlcuding 1000 in...,ya230501a-02.pl2,NaN,1,NaN,1
2,2003,ya230516,msn,2,1,1,NaN,NaN,1,2005,...,-3,-2,12500,2,probe stop is 1 mm lower than single elc,ya230516a-01.pl2,NaN,1,NaN,1
3,2004,ya230516,msn,3,1,2,NaN,NaN,1,2005,...,-3,-2,12500,2,NaN,ya230516a-01.pl2,NaN,1,NaN,1
4,2005,ya230516,msn,3,2,3,NaN,NaN,1,2005,...,-3,-2,12500,2,NaN,ya230516a-01.pl2,NaN,1,NaN,1


In [17]:
df.iloc[0].neural_data.keys()

dict_keys([1, 2, 3, 4, 5, 7, 9, 12, 15, 16, 17, 18, 19, 20, 21, 23, 25, 26, 27, 28, 29, 31, 33, 34, 35, 38])

In [18]:
cell_db.columns

Index(['cell_ID', 'session', 'cell_type', 'electrode', 'template',
       'maestro_ID', 'phy_id', 'phy_channel', 'file_begin', 'file_end',
       'fb_after_stablility', 'fe_after_stability', 'plexon_session', 'grade',
       'X', 'Y', 'depth_mm', 'is_continuous', 'comments', 'plex_sorted_file',
       'tmp', 'sorted', 'problem', 'synced_stability'],
      dtype='object')

In [19]:
# Function to extract session and trial number from filename
def parse_filename(filename):
    """
    Parse filename like 'fi210824a.0614' into components
    Returns: (session, plexon_session, trial_number)
    """
    parts = filename.split('.')
    if len(parts) != 2:
        return None, None, None
    
    prefix = parts[0]  # 'fi210824a'
    trial_num = parts[1]  # '0614'
    
    if len(prefix) < 9:  # minimum: 'fi' + 6 digits + 'a'
        return None, None, None
    
    session = prefix[:-1]  # 'fi210824' (remove plexon session letter)
    plexon_session = prefix[-1]  # 'a'
    
    return session, plexon_session, int(trial_num)

# Test the function
test_filename = df.iloc[0]['filename']
print(f"Test filename: {test_filename}")
session, plexon_session, trial_num = parse_filename(test_filename)
print(f"Parsed: session='{session}', plexon_session='{plexon_session}', trial_num={trial_num}")

# Check a few more examples
print("\nTesting with more filenames:")
for i in range(5):
    fname = df.iloc[i]['filename']
    session, plexon_session, trial_num = parse_filename(fname)
    print(f"{fname} -> session='{session}', plexon_session='{plexon_session}', trial_num={trial_num}")

Test filename: ya230528a.0525
Parsed: session='ya230528', plexon_session='a', trial_num=525

Testing with more filenames:
ya230528a.0525 -> session='ya230528', plexon_session='a', trial_num=525
ya230528a.0476 -> session='ya230528', plexon_session='a', trial_num=476
ya230528a.1504 -> session='ya230528', plexon_session='a', trial_num=1504
ya230528a.1499 -> session='ya230528', plexon_session='a', trial_num=1499
ya230528a.1105 -> session='ya230528', plexon_session='a', trial_num=1105


In [20]:
# Create the unified DataFrame
def create_unified_dataframe(trials_df, cells_df):
    """
    Create a unified DataFrame with one row per cell-trial combination.
    
    Parameters:
    -----------
    trials_df : pd.DataFrame
        DataFrame with trial data
    cells_df : pd.DataFrame  
        DataFrame with cell information
    
    Returns:
    --------
    pd.DataFrame : Unified DataFrame with cell-trial combinations
    """
    unified_data = []
    
    print("Creating unified DataFrame...")
    print(f"Processing {len(trials_df)} trials and {len(cells_df)} cells...")
    
    for trial_idx, trial_row in tqdm(trials_df.iterrows(), total=len(trials_df), desc="Processing trials"):
        # Parse filename to get session info
        filename = trial_row['filename']
        session, plexon_session, trial_num = parse_filename(filename)
        
        if session is None:
            print(f"Warning: Could not parse filename {filename}")
            continue
        
        # Get neural data for this trial
        neural_data = trial_row.get('neural_data', {})
        if not isinstance(neural_data, dict):
            neural_data = {}
        
        # Find cells that were active during this trial
        trial_cells = cells_df[
            (cells_df['session'] == session) & 
            (cells_df['plexon_session'] == plexon_session) &
            (cells_df['file_begin'] <= trial_num) & 
            (cells_df['file_end'] >= trial_num)
        ]
        
        # Create one row per cell for this trial
        for _, cell_row in trial_cells.iterrows():
            maestro_id = cell_row['maestro_ID']
            
            # Get neural data for this specific cell
            cell_neural_data = neural_data.get(maestro_id, [])
            
            # Create unified row
            unified_row = {
                # Cell information
                'cell_ID': cell_row['cell_ID'],
                'cell_type': cell_row['cell_type'], 
                'maestro_ID': maestro_id,
                'problem': cell_row['problem'],
                
                # Core trial information
                'filename': trial_row['filename'],
                'trial_name': trial_row['trial_name'],
                'reaction_time': trial_row['reaction_time'],
                'go_cue': trial_row['go_cue'],
                'stop_cue': trial_row['stop_cue'], 
                'trial_failed': trial_row['trial_failed'],
                'ssd_len': trial_row['ssd_len'],
                'ssd_number': trial_row['ssd_number'],
                'type': trial_row['type'],
                
                # Additional trial information
                'first_relevant_saccade': trial_row['first_relevant_saccade'],
                'segs_durations': trial_row['segs_durations'],
                'segs_times': trial_row['segs_times'],
                'trial_length': trial_row['trial_length'],
                'screen_rotation': trial_row['screen_rotation'],
                'saccades': trial_row['saccades'],
                'blinks': trial_row['blinks'],
                'dir': trial_row['dir'],
                
                # Neural data for this specific cell
                'neural_data': cell_neural_data,
                
                # Additional useful columns
                'session': session,
                'plexon_session': plexon_session,
                'trial_number': trial_num,
                'trial_session': trial_row['trial_session']
            }
            
            unified_data.append(unified_row)
    
    unified_df = pd.DataFrame(unified_data)
    print(f"\nUnified DataFrame created with {len(unified_df)} rows")
    print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
    print(f"Unique trials: {unified_df['filename'].nunique()}")
    
    return unified_df

# Create the unified DataFrame
unified_df = create_unified_dataframe(df, cell_db)
unified_df.head()

Creating unified DataFrame...
Processing 123178 trials and 3773 cells...


Processing trials:  15%|█▍        | 18458/123178 [00:57<05:26, 321.17it/s]



KeyboardInterrupt: 

In [ ]:
# Analyze the unified DataFrame with additional columns
print("=== UNIFIED DATAFRAME ANALYSIS (WITH ADDITIONAL COLUMNS) ===")
print(f"Shape: {unified_df.shape}")
print(f"Memory usage: {unified_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nColumn info:")
print(f"Total columns: {len(unified_df.columns)}")
print(f"Columns: {list(unified_df.columns)}")

print(f"\nData summary:")
print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
print(f"Unique trials: {unified_df['filename'].nunique()}")  
print(f"Unique sessions: {unified_df['session'].nunique()}")

print(f"\nCell types distribution:")
print(unified_df['cell_type'].value_counts())

print(f"\nTrial types in unified data:")
if 'type' in unified_df.columns:
    print(unified_df['type'].value_counts())

print(f"\nTrial name samples:")
trial_name_samples = unified_df['trial_name'].value_counts()
print(trial_name_samples.head(10))

print(f"\nNeural data statistics:")
neural_data_lengths = unified_df['neural_data'].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)
print(f"Mean spikes per trial: {neural_data_lengths.mean():.1f}")
print(f"Max spikes per trial: {neural_data_lengths.max()}")
print(f"Trials with no spikes: {(neural_data_lengths == 0).sum()}")

# Show sample rows with new columns
print(f"\nSample data with key columns:")
display_cols = ['cell_ID', 'cell_type', 'maestro_ID', 'filename', 'trial_name', 'type', 'reaction_time', 
                'first_relevant_saccade', 'trial_length', 'dir', 'problem']
available_cols = [col for col in display_cols if col in unified_df.columns]
print(f"Available columns: {available_cols}")
print(unified_df[available_cols].head(3))

# Check additional column data types and samples
print(f"\nAdditional column samples:")
additional_cols = ['segs_durations', 'segs_times', 'screen_rotation', 'saccades', 'blinks']
for col in additional_cols:
    if col in unified_df.columns:
        sample_val = unified_df[col].iloc[0]
        print(f"{col}: {type(sample_val).__name__} - {str(sample_val)[:100]}{'...' if len(str(sample_val)) > 100 else ''}")

=== UNIFIED DATAFRAME ANALYSIS (WITH ADDITIONAL COLUMNS) ===
Shape: (1875713, 26)
Memory usage: 2361.1 MB

Column info:
Total columns: 26
Columns: ['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'filename', 'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed', 'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade', 'segs_durations', 'segs_times', 'trial_length', 'screen_rotation', 'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session', 'trial_number', 'trial_session']

Data summary:
Unique cells: 2491
Unique trials: 45673
Unique sessions: 79

Cell types distribution:
cell_type
pu msn     1000077
msn         541169
hfdp        162156
lfd          37952
pu tan       30438
tan          30155
gpi          21850
fiber        16081
unknown      12045
lfdb          9701
fsn           5959
fef           5760
bd            1701
tan            543
ctx            126
Name: count, dtype: int64

Trial types in unified data:
type
GO      1037987
CONT     437213
S

In [ ]:
# Check for data quality and potential issues
print("=== DATA QUALITY CHECKS ===")

# Check for missing data
print("Missing values per column:")
missing_data = unified_df.isnull().sum()
print(missing_data[missing_data > 0])

# Check neural data distribution
print(f"\nNeural data statistics:")
neural_lengths = unified_df['neural_data'].apply(len)
print(f"Trials with neural data: {(neural_lengths > 0).sum()}/{len(unified_df)} ({(neural_lengths > 0).mean()*100:.1f}%)")

# Check problem cells
if 'problem' in unified_df.columns:
    problem_summary = unified_df['problem'].value_counts(dropna=False)
    print(f"\nProblem cell distribution:")
    print(problem_summary)

# Sample of actual neural data
print(f"\nSample neural data (first cell, first trial with data):")
sample_with_data = unified_df[neural_lengths > 0].iloc[0]
print(f"Cell ID: {sample_with_data['cell_ID']}")
print(f"Trial: {sample_with_data['trial_name']} ({sample_with_data['filename']})")
print(f"Neural data preview: {sample_with_data['neural_data'][:10]}...")  # First 10 spike times
print(f"Total spikes: {len(sample_with_data['neural_data'])}")

# Check trial distribution per cell
print(f"\nTrials per cell statistics:")
trials_per_cell = unified_df.groupby('cell_ID').size()
print(f"Mean trials per cell: {trials_per_cell.mean():.1f}")
print(f"Min trials per cell: {trials_per_cell.min()}")
print(f"Max trials per cell: {trials_per_cell.max()}")

# Show cells with most/least trials
print(f"\nCells with most trials:")
print(trials_per_cell.nlargest(5))
print(f"\nCells with fewest trials:")
print(trials_per_cell.nsmallest(5))

=== DATA QUALITY CHECKS ===
Missing values per column:
problem                   1872626
reaction_time              101902
stop_cue                  1037987
ssd_number                1037987
first_relevant_saccade     101902
saccades                     1977
blinks                    1486769
dtype: int64

Neural data statistics:
problem                   1872626
reaction_time              101902
stop_cue                  1037987
ssd_number                1037987
first_relevant_saccade     101902
saccades                     1977
blinks                    1486769
dtype: int64

Neural data statistics:
Trials with neural data: 1149148/1875713 (61.3%)

Problem cell distribution:
problem
NaN                         1872626
broken                         1707
check sorting with Mati         768
synch problem                   486
broken cell two peaks           126
Name: count, dtype: int64

Sample neural data (first cell, first trial with data):
Cell ID: 9866
Trial: CONT_L_SSD2 (fi210824a.0

In [ ]:
# Save the unified DataFrame
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
save_path.mkdir(exist_ok=True)

# Save as pickle for efficient loading
pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'
unified_df.to_pickle(pickle_file)
print(f"Unified DataFrame saved to: {pickle_file}")

# # Also save a CSV version for easy inspection (but this will be larger)
# csv_file = save_path / f'unified_{monkey}_cell_trial_data.csv'
# # For CSV, convert neural_data to string representation to avoid issues
# csv_df = unified_df.copy()
# csv_df['neural_data'] = csv_df['neural_data'].apply(lambda x: str(x) if x else "[]")
# csv_df.to_csv(csv_file, index=False)
# print(f"CSV version saved to: {csv_file}")

# print(f"\nFile sizes:")
# print(f"Pickle: {pickle_file.stat().st_size / 1e6:.1f} MB") 
# print(f"CSV: {csv_file.stat().st_size / 1e6:.1f} MB")

# print(f"\nDataFrame ready for neural analysis!")
# print(f"Use: pd.read_pickle('{pickle_file}') to load the unified data")

Unified DataFrame saved to: /home/barak/Projects/population_analysis/data/unified_cell_trial_data/unified_fiona_cell_trial_data.pkl
